### 1. Setup de Ambiente
Criação do schema **silver** e variáveis com os nomes do catálogo e schemas

In [0]:
catalog = "cinedata"
bronze_schema = "bronze"
silver_schema = "silver"

In [0]:
create_silver_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}"
spark.sql(create_silver_schema)

### 2. Tabela silver.tb_info_filmes (origem: tb_movies_info)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.pandas import DataFrame

# Dados da origem
df_bronze_movies_info = spark.table(f"{catalog}.{bronze_schema}.tb_movies_info")

# Colunas renomeadas, status_filme padronizado e data_lancamento convertida
df_silver_info_filmes = (
    df_bronze_movies_info
    .withColumnsRenamed({
        "id": "id_filme",
        "title": "titulo",
        "original_title": "titulo_original",
        "original_language": "idioma_original",
        "release_date": "data_lancamento",
        "runtime": "duracao_minutos",
        "status": "status_filme",
        "overview": "sinopse",
        "tagline": "frase_divulgacao"
    })
    .withColumn("status_filme", F.trim(F.col("status_filme")))
    .withColumn("status_filme", F.regexp_replace("status_filme", r"[-+]", ""))
    .withColumn("status_filme", F.initcap(F.col("status_filme")))
    .withColumn("status_filme", F.when(F.col("status_filme") == "Released", "Lançado")
                .when(F.col("status_filme") == "In Production", "Em Produção")
                .when(F.col("status_filme") == "Post Production", "Pós-Produção")
                .when(F.col("status_filme") == "Rumored", "Rumores")
                .when(F.col("status_filme") == "Planned", "Planejado")
                .when(F.col("status_filme") == "Canceled", "Cancelado")
                .otherwise("Não Informado"))
    .withColumn("data_lancamento", F.try_to_date(F.col("data_lancamento"), "yyyy-MM-dd"))
    .withColumn("duracao_minutos", F.col("duracao_minutos").try_cast("integer"))
)

# Utilizei o Window + row_number para manter apenas o registro mais atualizado com base em ingestion_datetime
window = Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))

df_silver_info_filmes = (
    df_silver_info_filmes
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("id_filme", "tconst", "titulo", "titulo_original", "idioma_original", "data_lancamento", "duracao_minutos", "status_filme", "sinopse", "frase_divulgacao")
)

# Coluna derivada ano_lancamento
df_silver_info_filmes = df_silver_info_filmes.withColumn("ano_lancamento", F.year(F.col("data_lancamento")))

# Salvar como tabela delta
df_silver_info_filmes.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_info_filmes")

### 3. Tabela silver.tb_financeiro_filmes (origem: tb_movies_financials)

In [0]:
# Dados da origem
df_bronze_movies_financials = spark.table(f"{catalog}.{bronze_schema}.tb_movies_financials")

# Cotação atualizada
df_cotacao_atual = spark.table(f"{catalog}.{bronze_schema}.tb_cotacao_dolar").orderBy(F.desc("dataHoraCotacao")).limit(1)
taxa = df_cotacao_atual.select("cotacaoCompra").collect()[0][0]

# Lista de valores textuais que indicam ausência na base de dados
ausentes = ["UNKNOWN", "NÃO INFORMADO", "N/A"]

df_silver_financeiro_filmes = (
    df_bronze_movies_financials
    .withColumn("revenue", F.trim(F.col("revenue")))
    .withColumn("budget", F.trim(F.col("budget")))
    .withColumn("revenue", F.when(F.upper(F.col("revenue")).isin(ausentes), None).otherwise(F.col("revenue")))
    .withColumn("budget", F.when(F.upper(F.col("budget")).isin(ausentes), None).otherwise(F.col("budget")))
    .withColumn("revenue", F.regexp_replace(F.regexp_replace(F.col("revenue"), r"[M]", "00000"), r"[K]", "00"))
    .withColumn("budget", F.regexp_replace(F.regexp_replace("budget", r"[M]", "00000"), r"[K]", "00"))
    .withColumn("revenue", F.regexp_replace("revenue", r"[^\d]", ""))
    .withColumn("budget", F.regexp_replace("budget", r"[^\d]", ""))
    .withColumn("revenue", F.col("revenue").try_cast("decimal(18,2)"))
    .withColumn("budget", F.col("budget").try_cast("decimal(18,2)"))
    .withColumn("revenue", F.when(F.col("revenue") <= 0, None).otherwise(F.col("revenue")))
    .withColumn("budget", F.when(F.col("budget") <= 0, None).otherwise(F.col("budget")))
    .withColumn("orcamento_brl", F.col("budget") * taxa)
    .withColumn("receita_brl", F.col("revenue") * taxa)
    .withColumn("orcamento_brl", F.round(F.col("orcamento_brl"), 2))
    .withColumn("receita_brl", F.round(F.col("receita_brl"), 2))
    .withColumn("lucro_brl", F.coalesce(F.col("receita_brl"), F.lit(0)) - F.coalesce(F.col("orcamento_brl"), F.lit(0)))
    .withColumn("lucro_usd", F.coalesce(F.col("revenue"), F.lit(0)) - F.coalesce(F.col("budget"), F.lit(0)))
    .withColumn("lucro_percentual", F.when(F.col("orcamento_brl") > 0, F.round(F.col("lucro_brl") / F.col("orcamento_brl") * 100, 2)).otherwise(None))
    .withColumnsRenamed({
        "id": "id_filme",
        "budget": "orcamento_usd",
        "revenue": "receita_usd"
    })
)

window = Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))

df_silver_financeiro_filmes = (
    df_silver_financeiro_filmes
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("id_filme", "orcamento_usd", "orcamento_brl", "receita_usd", "receita_brl", "lucro_brl", "lucro_usd", "lucro_percentual")
)

df_silver_financeiro_filmes.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_financeiro_filmes")

### 4. Tabela silver.tb_metricas_engajamento (origem: tb_movies_metrics)

In [0]:
# Funções auxiliares para usar com transform() e lista de colunas não aplicáveis
def trim_func(df):
    for c in df.columns:
        if c == "id":
            continue
        df = df.withColumns({
            c: F.trim(F.col(c))
        })
    return df

def regex_func(df):
    for c in df.columns:
        if c == "id":
            continue
        df = df.withColumns({
            c: F.regexp_replace(F.regexp_replace(F.col(c), r"[^\d,\.]", ""), r"[,]", ".")
        })
    return df

def round_func(df, ignored_cols:list[str]=[]):
    for c in df.columns:
        if c in ["id", "vote_count", "numVotes"]:
            continue
        df = df.withColumns({
            c: F.round(F.col(c).try_cast("double"), 2)
        })
    return df

def int_safe_cast_func(df, ignored_cols:list[str]=[]):
    for c in df.columns:
        if c in ["id", "popularity", "vote_average", "averageRating"]:
            continue
        df = df.withColumns({
            c: F.col(c).try_cast("int")
        })
    return df

def avg_fix_func(df, ignored_cols:list[str]=[]):
    for c in df.columns:
        if c in ["id", "popularity", "vote_count", "numVotes"]:
            continue
        df = df.withColumns({
            c: F.when((F.col(c) < 0) | (F.col(c) > 10), None).otherwise(F.col(c))
        })
    return df

def negative_val_fix(df, ignored_cols:list[str]=[]):
    for c in df.columns:
        if c in ["id", "popularity", "vote_count", "numVotes"]:
            continue
        df = df.withColumns({
            c: F.when(F.col(c) < 0, None).otherwise(F.col(c))
        })
    return df

In [0]:
# Dados da origem
df_bronze_movies_metrics = spark.table(f"{catalog}.{bronze_schema}.tb_movies_metrics")

# Aplicação das funções de transformação e colunas renomeadas
df_silver_metricas_engajamento = (
    df_bronze_movies_metrics
    .transform(trim_func)
    .transform(regex_func)
    .transform(round_func)
    .transform(int_safe_cast_func)
    .transform(avg_fix_func)
    .transform(negative_val_fix)
    .withColumnsRenamed({
        "id": "id_filme",
        "popularity": "popularidade",
        "vote_average": "nota_media_tmdb",
        "vote_count": "qtd_votos_tmdb",
        "averageRating": "nota_media_imdb",
        "numVotes": "qtd_votos_imdb"
    })
)

window = Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))

df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)

# Salva como tabela delta
df_silver_metricas_engajamento.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_metricas_engajamento")

### 5. Tabela silver.tb_avaliacoes_usuarios (origem: tb_movies_reviews)

In [0]:
# Dados da origem
df_bronze_movies_reviews = spark.table(f"{catalog}.{bronze_schema}.tb_movies_reviews")

# Apliquei as regras de negócio antes de fazer dedup
df_silver_avaliacoes_usuarios = (
    df_bronze_movies_reviews
    .withColumn("comentario", F.when((F.col("comentario").isNull()) | (F.trim(F.col("comentario")) == ""), "Sem comentário").otherwise(F.col("comentario")))
    .withColumn("nota", F.when((F.col("nota") < 0) | (F.col("nota") > 10), None).otherwise(F.col("nota")))
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .withColumnsRenamed({
        "id": "id_filme",
        "nome": "nome_usuario",
        "nota": "nota_usuario",
        "comentario": "comentario_usuario"
    })
).select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")

# Salva como tabela delta
df_silver_avaliacoes_usuarios.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_avaliacoes_usuarios")

### 6. Tabela silver.tb_generos (origem: tb_credits_and_tags, coluna genres)

In [0]:
# Dados da origem
df_bronze_credits_and_tags = spark.table(f"{catalog}.{bronze_schema}.tb_credits_and_tags")

window = Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))

# Limpeza, padronização de split e explode da coluna 'genres' da origem
df_silver_generos = (
    df_bronze_credits_and_tags
    .withColumn("genres", F.regexp_replace("genres", r"[\|;]", ","))
    .withColumn("genero", F.explode_outer(F.split(F.col("genres"), ",")))
    .withColumn("genero", F.trim(F.col("genero")))
    .filter(~F.col("genero").rlike(r"[\d]"))
    .filter(F.length(F.col("genero")) <= 25)
    .filter(F.col("genero").rlike(r"^[A-Za-zÀ-ÿ\s\-]+$"))
    .withColumnRenamed("id", "id_filme")
)

df_silver_generos = (
    df_silver_generos
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("id_filme", "genero")
)

# Padrão de exclusão de column shift com base na contagem de registros por gênero
dominio = (
    df_silver_generos
    .groupBy("genero").count()
    .filter(F.col("count") >= 20)
    .select("genero")
)

# Join com os gêneros de domínio filtrados
df_silver_generos = df_silver_generos.join(dominio, on="genero", how="inner")

# Salva como tabela delta
df_silver_generos.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_generos")

### 7. Tabela silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers, production_companies)

In [0]:
# Função auxiliar para extrair dados de uma coluna e rotular a entidade corretamente
def extrair(df, coluna_origem, rotulo):
    return (
        df
        .select("id", coluna_origem)
        .withColumns({
            "nome": F.explode(F.split(F.col(coluna_origem), ",")),
            "tipo_entidade": F.lit(rotulo)
        })
        .select(F.col("id").alias("id_filme"), "nome", "tipo_entidade")
    )

In [0]:
# Lista de valores textuais que indicam ausência na base de dados
ausentes = ["NENHUM", "N/A", "[]", ".", ""]

# Dados extraídos, rotulados e unidos de acordo com o tipo_entidade
df_silver_pessoas_empresas = (
    extrair(df_bronze_credits_and_tags, "cast", "Ator")
    .unionByName(extrair(df_bronze_credits_and_tags, "directors", "Diretor"))
    .unionByName(extrair(df_bronze_credits_and_tags, "writers", "Roteirista"))
    .unionByName(extrair(df_bronze_credits_and_tags, "production_companies", "Produtora"))
)

# Limpeza e dedup, com padrão regex de alfabetos e dígitos amplos
df_silver_pessoas_empresas = (
    df_silver_pessoas_empresas
    .withColumn("nome", F.when(F.upper(F.col("nome")).isin(ausentes), None).otherwise(F.col("nome")))
    .withColumn("nome", F.regexp_replace("nome", r"[^\p{L}\p{N}\s\-\.&']", ""))
    .withColumn("nome", F.regexp_replace("nome", r"\s+", " "))
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("nome", F.when(F.col("nome") == F.upper(F.col("nome")), F.col("nome")).otherwise(F.initcap(F.col("nome"))))
    .filter((F.col("nome").isNotNull()) & (F.col("nome") != "") & (~F.col("nome").rlike(r"^[\d\.\s]+$")))
    .dropDuplicates(["id_filme", "nome", "tipo_entidade"])
)

# Padrão de exclusão de column shift com base nos países de domínio filtrados
dominio_countries = (
    df_bronze_credits_and_tags
    .select("production_countries")
    .withColumn("production_countries", F.regexp_replace("production_countries", r"[\|;]", ","))
    .withColumn("production_countries", F.explode(F.split(F.col("production_countries"), ",")))
    .withColumn("production_countries", F.trim(F.col("production_countries")))
    .withColumn("production_countries", F.when(F.col("production_countries") == F.upper(F.col("production_countries")), F.col("production_countries")).otherwise(F.initcap(F.col("production_countries"))))
    .filter(~F.col("production_countries").rlike(r"[\d]"))
    .dropDuplicates(["production_countries"])
    .withColumnRenamed("production_countries", "nome")
)

df_silver_pessoas_empresas = df_silver_pessoas_empresas.join(
    dominio_countries,
    on="nome",
    how="left_anti"
)

# Exclusão de column shift com base nos gêneros de domínio filtrados
df_silver_pessoas_empresas = df_silver_pessoas_empresas.join(
    df_silver_generos.select(F.col("genero").alias("nome")).distinct(),
    on="nome",
    how="left_anti"
)

# Salva como tabela delta
df_silver_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_pessoas_empresas")

### 8. Tabela silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)

In [0]:
# Dados da origem
df_bronze_cotacao_dolar = spark.table(f"{catalog}.{bronze_schema}.tb_cotacao_dolar")
df_bronze_cotacao_dolar = df_bronze_cotacao_dolar.withColumn("dataHoraCotacao", F.col("dataHoraCotacao").cast("date"))

# Sequência temporal de datas (1 semana)
df_sequencia_dias = (
    spark.range(1)
    .select(F.explode(F.sequence(
        F.to_date(F.lit(dbutils.widgets.get("data_inicio")), "MM-dd-yyyy"),
        F.to_date(F.lit(dbutils.widgets.get("data_fim")), "MM-dd-yyyy"),
        F.expr("INTERVAL 1 DAY"))).alias("dataHoraCotacao"))
)

window_dedup = Window.partitionBy("dataHoraCotacao").orderBy(F.desc("ingestion_datetime"))

df_silver_cotacao_dolar = (
    df_bronze_cotacao_dolar
    .withColumn("rn", F.row_number().over(window_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("cotacaoCompra", "dataHoraCotacao")
)

df_silver_cotacao_dolar = (
    df_sequencia_dias
    .join(df_silver_cotacao_dolar, on="dataHoraCotacao", how="left")
)

# Forward fill usando window function
window_forward_fill = (Window.orderBy("dataHoraCotacao")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

df_silver_cotacao_dolar = (
    df_silver_cotacao_dolar.withColumn("cotacaoCompra", F.last(F.col("cotacaoCompra"), ignorenulls=True).over(window_forward_fill))
)

# Salvo como tabela delta
df_silver_cotacao_dolar.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.tb_cotacao_dolar")